# A1 | 07 — Fechar códigos, protocolo e coorte

**Pessoa.** Converter a pergunta de pesquisa em regras de inclusão, exclusão e desfecho.

**Saída:** config de decisões v0.1 (`configs/decisoes.yaml`), tabela de códigos, coorte provisória e waterfall.
**Revisão:** B confere categorias e gráficos; C implementa e testa as regras.

| Passo do plano | Onde está neste notebook |
|---|---|
| 1–3. Fichas/dicionários, pesquisar HOSPITAL, DT_INTERNA, EVOLUCAO, TP_IDADE e códigos; registrar tabela | Seção 1 (manual) + seção 2 |
| 4. Ponto de previsão em DT_INTERNA e campos posteriores | Seção 3 |
| 5–6. Coorte estrita, alvo e categorias separadas | Seções 4 e 6 |
| 7–9. N antes/depois, amostras das exclusões, waterfall | Seções 4 e 5 |
| 10. Atualizar tabela de decisões | Seção 7 |

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa_a"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "minimas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
SUPERVISED_CONFIG = "configs/supervised.yaml"   # features/alvo/coluna_tempo definidos pelo C
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A1")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

## 1. Fichas e dicionários (manual)
Busque exatamente por: *"SIVEP-Gripe dicionário de dados HOSPITAL EVOLUCAO DT_INTERNA"*, *"SIVEP-Gripe ficha de investigação SRAG hospitalização evolução"*, *"STROBE cohort flow diagram observational study"*.
Para cada campo (HOSPITAL, DT_INTERNA, EVOLUCAO, TP_IDADE, sim/não/ignorado) registre: definição, código, ano, regra de salto, tratamento e evidência — na tabela de decisões (seção 7).

- [ ] Fichas/dicionários dos anos usados localizados
- [ ] Códigos de HOSPITAL, EVOLUCAO, TP_IDADE confirmados por ano

## 2. Colunas essenciais e tabela de códigos

In [ ]:
v.a1_columns(df, cfg, rep)
v.a1_code_tables(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a1_tabela_codigos.csv").head(40))

## 3. Ponto de previsão e campos posteriores
O ponto de previsão proposto é a **internação** (`DT_INTERNA`). Tudo o que só existe depois dela é candidato a vazamento.

In [ ]:
print("Ponto de previsão:", cfg["cols"]["dt_interna"])
posteriores = [c for c in cfg["forbidden_vars"] if v.canon(c) in df.columns]
print("Campos potencialmente posteriores presentes na base:")
print(posteriores)

## 4. Coorte estrita — waterfall
Filtros em sequência: hospitalização confirmada → data válida → período → evolução encerrada → alvo binário.

In [ ]:
wf = v.a1_waterfall(df, cfg, rep)
display(wf)

## 5. Inspeção das exclusões
Confirme que cada filtro se comporta como a regra escrita (as amostras vêm de `a1_amostra_exclusoes_NN.csv`).

In [ ]:
for f in sorted(Path(OUT).glob("a1_amostra_exclusoes_*.csv")):
    print("\n==", f.name)
    display(pd.read_csv(f).head(5))

## 6. Alvo e categorias separadas
`EVOLUCAO=2` → óbito (1); `EVOLUCAO=1` → cura (0). **Óbito por outras causas (3) fica separado e evolução desconhecida (9/ausente) nunca vira cura.**

In [ ]:
v.a1_target(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a1_alvo_por_ano.csv").head(40))

## 7. Tabela de decisões
Ajuste o texto, o `status` (`proposta` → `aprovada`) e o impacto após conferir no dicionário. A mesma tabela alimenta B e C.

In [ ]:
E = "A1"
v.log_decision(OUT, "D-01", "Ponto de previsão", "DT_INTERNA (admissão); campos posteriores bloqueados", E,
               impacto="Define quais variáveis podem entrar", evidencia="Dicionário SIVEP-Gripe (completar)")
v.log_decision(OUT, "D-02", "Coorte estrita", "HOSPITAL=1, DT_INTERNA válida, período analítico, evolução encerrada", E,
               impacto="Ver waterfall (a1_waterfall.csv)")
v.log_decision(OUT, "D-03", "Alvo principal", "Óbito SRAG (EVOLUCAO=2) vs cura (EVOLUCAO=1)", E,
               impacto="Alinhar target.positivos do supervised.yaml (G1)")
v.log_decision(OUT, "D-04", "Evolução desconhecida e óbito outras causas",
               "EVOLUCAO=9/ausente e 3 saem do alvo binário; nunca viram cura", E,
               impacto="Exigir filtro EVOLUCAO in {1,2} no loader do C")
v.log_decision(OUT, "D-12", "Features derivadas (criadas no dataset analítico)",
               f"idade_anos (TP_IDADE), faixa_etaria, intervalo sintomas->internação (inválido se ausente/negativo/>{cfg['derivadas']['intervalo_max_dias']} dias) "
               "+ flag, sin/cos da semana ISO da internação", E,
               impacto="Só informação da admissão; o C2 as encontra no dataset (supervised.yaml)")
v.log_decision(OUT, "D-05", "Período analítico",
               f"Desenvolvimento {cfg['dev_years']}; holdout {cfg['holdout_years']}", E,
               impacto="Exclui anos fora do período; alinhar G2")
display(pd.read_csv(Path(OUT) / "tabela_decisoes.csv"))

## 8. Exportar a população analítica (v0.1, provisória)
O notebook **C2 exige** `data/analytical/dataset_supervisionado.parquet`. Esta célula o gera a partir da coorte, usando as features, o alvo e a coluna de tempo do `supervised.yaml`. Sai ordenado por tempo e **sem** colunas pós-admissão além das features pedidas. Avise o C quando atualizar.

In [ ]:
info = v.a1_export_dataset(df, cfg, rep, SUPERVISED_CONFIG, versao="0.1", excluir_proibidas=False)
print("Arquivo:", info["arquivo"], "|", f"{info['n_linhas']:,} linhas")
print("Anos:", info["anos"])
print("Features proibidas no dataset:", info["features_proibidas_no_dataset"] or "nenhuma")
print("Features ausentes na base:", info["features_ausentes_na_base"] or "nenhuma")
print("Linhas descartadas por data de tempo inválida:", info["descartadas_data_tempo_invalida"])

# Confere o ARQUIVO gravado contra o que o C2 consome (colunas, datas, alvo, check_leakage)
v.validar_contrato_c2(info["arquivo"], SUPERVISED_CONFIG, rep)

## 9. Fechamento

In [ ]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

### Checklist de saída
- [ ] Config de decisões v0.1 atualizada (`configs/decisoes.yaml`)
- [ ] Tabela de códigos e tabela de decisões revisadas
- [ ] Waterfall com N excluído/restante/% de cada filtro
- [ ] Amostras das exclusões inspecionadas
- [ ] `data/analytical/dataset_supervisionado.parquet` (v0.1) gerado e C avisado
- [ ] B e C avisados das regras (categorias, alvo, ponto de previsão)